# RDIM reaction network (Catalyst)
Minimal Catalyst definition of the homogeneous RDIM reactions and a parameter set matching `c_impl/analysis/find_equilibria.py`.

In [11]:
using Catalyst, DifferentialEquations, Plots
import HomotopyContinuation
using StructuralIdentifiability
using ModelingToolkit
using CairoMakie
using LaTeXStrings
using Symbolics
using GenericLinearAlgebra
using Latexify

CairoMakie.activate!()


Welcome to Nemo version 0.49.5

Nemo comes with absolutely no warranty whatsoever


In [2]:
# We define time as the independent variable
@variables t;
# For Catalyst, we need to define the species and parameters of the system
@species C6i(t) C12i(t) C6e(t) C12e(t) TETR(t) LACI(t) LUXI(t) LASI(t) RFP(t) GFP(t);
@parameters K_1R K_1S K_2R K_2S K_GR K_GS k_tx k_tr d_m d_p QSSA k_HSL Km_HSL d_HSL Dm_HSL d_HSLi d_HSLe KR_TET KR_LAC n_TET n_LAC n_LUX n_LAS leak_LUX leak_LAS;

In [3]:
# We define the reaction network
SIMPLIFIED_REACTION_NETWORK = @reaction_network begin
    @species C6i(t) C12i(t) C6e(t) C12e(t) TETR(t) LACI(t) LUXI(t) LASI(t) RFP(t) GFP(t)
    @parameters K_1R K_1S K_2R K_2S K_GR K_GS k_tx k_tr d_m d_p QSSA k_HSL Km_HSL Dm_HSL d_HSLi d_HSLe KR_TET KR_LAC n_TET n_LAC n_LUX n_LAS leak_LUX leak_LAS
    # C6i and C12i reactions
    # -----------------------
    # Production of C6i and C12i
    (k_HSL*LUXI)/(Km_HSL + LUXI), 0 --> C6i
    (k_HSL*LASI)/(Km_HSL + LASI), 0 --> C12i
    # Diffusion through the membrane (C6i and C12i)
    (Dm_HSL, Dm_HSL), C6i <--> C6e
    (Dm_HSL, Dm_HSL), C12i <--> C12e
    # Degradation of HSL (C6i and C12i)
    d_HSLi, C6i --> 0
    d_HSLi, C12i --> 0
    # C6e and C12e reactions (diffusion through the membrane is included above)
    d_HSLe, C6e --> 0
    d_HSLe, C12e --> 0
    # TETR and LACI reactions
    # -----------------------
    # Production of TETR and LACI
    (((((K_1S * C12i)/(1 + K_1S * C12i))^n_LAS)*(QSSA^2 * K_2S * K_GS * k_tx))/(1 + (KR_LAC * LACI)^n_LAC + ((((K_1S * C12i)/(1 + K_1S * C12i))^n_LAS)*(QSSA^2 * K_2S * K_GS))) + leak_LAS) * (k_tr/d_m), 0 --> TETR
    (((((K_1R * C6i)/(1 + K_1R * C6i))^n_LUX)*(QSSA^2 * K_2R * K_GR * k_tx))/(1 + (KR_TET * TETR)^n_TET + ((((K_1R * C6i)/(1 + K_1R * C6i))^n_LUX)*(QSSA^2 * K_2R * K_GR))) + leak_LUX) * (k_tr/d_m), 0 --> LACI
    # Degradation of TETR and LACI
    d_p, TETR --> 0
    d_p, LACI --> 0
    # LUXI and LASI reactions
    # -----------------------
    # Production of LUXI and LASI
    (((((K_1R * C6i)/(1 + K_1R * C6i))^n_LUX)*(QSSA^2 * K_2R * K_GR * k_tx))/(1 + (KR_TET * TETR)^n_TET + ((((K_1R * C6i)/(1 + K_1R * C6i))^n_LUX)*(QSSA^2 * K_2R * K_GR))) + leak_LUX) * (k_tr/d_m), 0 --> LUXI
    (((((K_1S * C12i)/(1 + K_1S * C12i))^n_LAS)*(QSSA^2 * K_2S * K_GS * k_tx))/(1 + (KR_LAC * LACI)^n_LAC + ((((K_1S * C12i)/(1 + K_1S * C12i))^n_LAS)*(QSSA^2 * K_2S * K_GS))) + leak_LAS) * (k_tr/d_m), 0 --> LASI
    # Degradation of LUXI and LASI
    d_p, LUXI --> 0
    d_p, LASI --> 0
    # RFP and GFP reactions
    # -----------------------
    # Production of RFP and GFP
    (((((K_1R * C6i)/(1 + K_1R * C6i))^n_LUX)*(QSSA^2 * K_2R * K_GR * k_tx))/(1 + (KR_TET * TETR)^n_TET + ((((K_1R * C6i)/(1 + K_1R * C6i))^n_LUX)*(QSSA^2 * K_2R * K_GR))) + leak_LUX) * (k_tr/d_m), 0 --> RFP
    (((((K_1S * C12i)/(1 + K_1S * C12i))^n_LAS)*(QSSA^2 * K_2S * K_GS * k_tx))/(1 + (KR_LAC * LACI)^n_LAC + ((((K_1S * C12i)/(1 + K_1S * C12i))^n_LAS)*(QSSA^2 * K_2S * K_GS))) + leak_LAS) * (k_tr/d_m), 0 --> GFP
    # Degradation of RFP and GFP
    d_p, RFP --> 0
    d_p, GFP --> 0
end;

In [15]:
# Parameter set with names aligned to the reaction-network parameters
# (values mirror c_impl/analysis/find_equilibria.py)
ps = Dict(
    :k_tx   => 1e-1,
    :k_tr   => 5.0,
    :d_m    => 5e-1,   # was D_MRNA
    :d_p    => 5e-2,
    :QSSA   => (1e-1 * 5.0) / (5e-1 * 5e-2),  # (k_tx*k_tr)/(d_m*d_p)

    :K_1R   => 1/0.1,
    :K_1S   => 1/0.1,
    :K_2R   => 1/0.02,
    :K_2S   => 1/0.02,
    :K_GR   => 1/0.10,
    :K_GS   => 1/0.10,

    :k_HSL  => 0.04,   # was K_LUXI/K_LASI (same value)
    :Km_HSL => 10.0,   # was KM_ENZ
    :Dm_HSL => 0.1,    # was DM
    :d_HSLi => 0.01,
    :d_HSLe => 0.001,

    :KR_TET => 1/0.001,
    :KR_LAC => 1/0.100,
    :n_TET  => 2.0,    # was n_REP1
    :n_LAC  => 4.0,    # was n_REP2
    :n_LUX  => 2.0,    # was n_R
    :n_LAS  => 2.0,    # was n_S
    :leak_LUX => 5e-5,   # was b_0
    :leak_LAS => 2.5e-4, # was b_1
)
SIMPLIFIED_REACTION_NETWORK

Model ##ReactionSystem#282:
Unknowns (10): see unknowns(##ReactionSystem#282)
  C6i(t)
  C12i(t)
  C6e(t)
  C12e(t)
  TETR(t)
  LACI(t)
  ⋮
Parameters (24): see parameters(##ReactionSystem#282)
  K_1R
  K_1S
  K_2R
  K_2S
  K_GR
  K_GS
  ⋮

## Scan Dm_HSL values and compute steady states
Loops over the ten Dm coefficients, finds steady states with HomotopyContinuation, keeps real non‑negative solutions, and reports C6e, C12e, GFP, RFP.

In [16]:
u0_LOW = [:C6i => 0.0, :C12i => 0.0, :C6e => 0.0, :C12e => 0.0, :TETR => 0.0, :LACI => 0.0, :LUXI => 0.0, :LASI => 0.0, :RFP => 0.0, :GFP => 0.0]
u0_HIGH = [:C6i => 0.0, :C12i => 0.0, :C6e => 1.0, :C12e => 0.0, :TETR => 0.0, :LACI => 0.0, :LUXI => 0.0, :LASI => 0.0, :RFP => 0.0, :GFP => 0.0]
tspan = (0.0, 1440.0)

# We create the ODE problems for both initial conditions
oprob_LOW = ODEProblem(SIMPLIFIED_REACTION_NETWORK, u0_LOW, tspan, ps; sparse = true, combinatoric_ratelaws = false);
oprob_HIGH = ODEProblem(SIMPLIFIED_REACTION_NETWORK, u0_HIGH, tspan, ps; sparse = true, combinatoric_ratelaws = false);

# Use stiff solvers for numerical integration
sol_LOW = solve(oprob_LOW, Tsit5());
sol_HIGH = solve(oprob_HIGH, Tsit5());

In [17]:
using LinearAlgebra
Dm_grid = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0, 3.0, 5.0, 10.0]

function ss_for_dm(dm; atol=1e-10, rtol=1e-10)
    pmap = copy(ps)
    pmap[:Dm_HSL] = dm
    sols = hc_steady_states(SIMPLIFIED_REACTION_NETWORK, pmap; u0_LOW)
    # filter real, non-negative
    real_sols = [sol for sol in sols if maximum(abs.(imag.(sol))) < 1e-8 && all(real.(sol) .>= -1e-9)]
    return [real.(sol) for sol in real_sols]
end


ss_for_dm (generic function with 1 method)

In [18]:
# run scan and collect summary
species = Catalyst.species(SIMPLIFIED_REACTION_NETWORK)
pick_idxs = Dict(:C6e => findfirst(==(Symbol("C6e")), species),
                 :C12e => findfirst(==(Symbol("C12e")), species),
                 :GFP => findfirst(==(Symbol("GFP")), species),
                 :RFP => findfirst(==(Symbol("RFP")), species))

scan_results = []
for dm in Dm_grid
    sols = ss_for_dm(dm)
    push!(scan_results, (dm=dm, sols=sols))
end

println("Dm value -> #steady states kept")
for r in scan_results
    println(r.dm, " -> ", length(r.sols))
end


Tracking 209 paths... 100%|█████████████████████████████| Time: 0:00:02
                   # paths tracked: 209
   # non-singular solutions (real): 17 (3)
       # singular endpoints (real): 52 (0)
          # total solutions (real): 69 (3)
Dm value -> #steady states kept
0.01 -> 3
0.05 -> 3
0.1 -> 3
0.25 -> 3
0.5 -> 3
0.75 -> 3
1.0 -> 3
3.0 -> 3
5.0 -> 3
10.0 -> 3


In [20]:
scan_results

10-element Vector{Any}:
 (dm = 0.01, sols = [[2.433837499885099, 0.018264924268372697, 2.2125795453500903, 0.016604476607611544, 0.050062809849351075, 19.741887728361352, 19.741887728361352, 0.05006280984935107, 19.74188772836135, 0.050062809849351075], [1.058988404740223, 0.2749942674708169, 0.9627167315820208, 0.24999478860983357, 0.8107925386190508, 4.061039355207443, 4.061039355207443, 0.8107925386190508, 4.061039355207443, 0.8107925386190508], [0.0036675609598269335, 2.446473692373252, 0.0033341463271153943, 2.224066993066593, 20.04989164758917, 0.010012453877242249, 0.010012453877242247, 20.04989164758917, 0.010012453877242249, 20.049891647589174]])
 (dm = 0.05, sols = [[1.0514692812853066, 0.273189132245251, 1.0308522365542219, 0.2678324825933833, 0.8107301116457553, 4.057552699376802, 4.057552699376802, 0.8107301116457553, 4.057552699376803, 0.8107301116457554], [2.4180280374532854, 0.01814607260430015, 2.370615722993417, 0.017790267259117795, 0.050062121511479235, 19.741758477

In [21]:
# print selected species for each steady state
for r in scan_results
    println("
Dm = ", r.dm)
    if isempty(r.sols)
        println("  (no real non-negative steady states)")
        continue
    end
    for (i, sol) in enumerate(r.sols)
        c6e = sol[3]
        c12e = sol[4]
        gfp = sol[10]
        rfp = sol[9]
        println("  ss", i, ": C6e=", c6e, "  C12e=", c12e, "  GFP=", gfp, "  RFP=", rfp)
    end
end



Dm = 0.01
  ss1: C6e=2.2125795453500903  C12e=0.016604476607611544  GFP=0.050062809849351075  RFP=19.74188772836135
  ss2: C6e=0.9627167315820208  C12e=0.24999478860983357  GFP=0.8107925386190508  RFP=4.061039355207443
  ss3: C6e=0.0033341463271153943  C12e=2.224066993066593  GFP=20.049891647589174  RFP=0.010012453877242249

Dm = 0.05
  ss1: C6e=1.0308522365542219  C12e=0.2678324825933833  GFP=0.8107301116457554  RFP=4.057552699376803
  ss2: C6e=2.370615722993417  C12e=0.017790267259117795  GFP=0.05006212151147924  RFP=19.741758477040783
  ss3: C6e=0.0035722440572894506  C12e=2.3829289189426777  GFP=20.049891591962112  RFP=0.010012297944523466

Dm = 0.1
  ss1: C6e=2.391971904099669  C12e=0.017950506816730785  GFP=0.050062028679841726  RFP=19.741740869597486
  ss2: C6e=0.0036044188831738278  C12e=2.404396746738997  GFP=20.049891584388018  RFP=0.010012276942928658
  ss3: C6e=1.0400527685150442  C12e=0.27024274043404334  GFP=0.8107215255928834  RFP=4.057078776357152

Dm = 0.25
  ss1: C6e

In [23]:
using OrdinaryDiffEq, SteadyStateDiffEq

In [31]:
# steady-state delta(GFP-RFP) at a given external C6e
function delta_at_c6e(c6e; dm=0.1)
    p = merge(ps, Dict(:Dm_HSL => dm))
    u0 = Dict(u0_LOW)           # start from the LOW template
    u0[:C6e] = c6e
    u0[:C12e] = 1e-6            # clamp external C12
    prob = ODEProblem(SIMPLIFIED_REACTION_NETWORK, u0, (0.0, 1440.0), p; sparse=true)
    sol = solve(prob, Rodas5P(); reltol=1e-6, abstol=1e-6, save_everystep=false)
    u = sol[end]
    return u[10] - u[9], u
end

delta_at_c6e (generic function with 1 method)

In [32]:
first_grid = range(1e-6, 1e-5; length=100)
second_grid = range(1e-5, 1e-4; length=100)
third_grid = range(1e-4, 1e-3; length=1000)
fourth_grid = range(1e-3, 1e-2; length=1000)
fifth_grid = range(1e-2, 1e-1; length=1000)
sixth_grid = range(1e-1, 1.0; length=100)

# Join the grids
c6_grid = vcat(collect(first_grid), collect(second_grid), collect(third_grid), collect(fourth_grid), collect(fifth_grid), collect(sixth_grid))
# Remove possible duplicates
c6_grid = unique(c6_grid)

dm_choice = 0.1  # pick one Dm_HSL to analyze
vals = [(c, delta_at_c6e(c; dm=dm_choice)[1]) for c in c6_grid]
sign_changes = findall(i -> vals[i][2]*vals[i+1][2] < 0, 1:length(vals)-1)
println("Sign-change intervals: ", [(vals[i][1], vals[i+1][1]) for i in sign_changes])

Sign-change intervals: [(0.0012432432432432433, 0.0012522522522522522)]


In [35]:
function bisection_c6e(a, b; dm=dm_choice, tol=1e-8, maxiter=10000)
    fa, _ = delta_at_c6e(a; dm=dm)
    fb, _ = delta_at_c6e(b; dm=dm)
    for _ in 1:maxiter
        m = (a+b)/2
        fm, _ = delta_at_c6e(m; dm=dm)
        if abs(fm) < tol || (b-a) < tol
            return m, fm
        end
        if fa*fm <= 0
            b, fb = m, fm
        else
            a, fa = m, fm
        end
    end
    return (a+b)/2, (fa+fb)/2
end

if !isempty(sign_changes)
    a, b = vals[sign_changes[1]][1], vals[sign_changes[1]+1][1]
    c6_star, delta_star = bisection_c6e(a, b; dm=dm_choice)
    _, u_star = delta_at_c6e(c6_star; dm=dm_choice)
    println("C6e* ≈ ", c6_star, " where GFP ≈ RFP (Δ=", delta_star, ")")
    println("Steady state at C6e*: GFP=", u_star[10], " RFP=", u_star[9])
else
    println("No sign change found on grid; widen the C6e range.")
end

C6e* ≈ 0.0012480776675112613 where GFP ≈ RFP (Δ=-19.691665266679877)
Steady state at C6e*: GFP=0.05006208148734622 RFP=19.741727348167224


In [37]:
# steady-state Δ = GFP − RFP at a given C6e, with C12e clamped to 1e-6
function delta_at_c6e(c6e; dm=0.1, tmax=1440.0)
    p = merge(ps, Dict(:Dm_HSL => dm))
    u0 = Dict(u0_LOW)          # start from LOW ICs
    u0[:C6e] = c6e
    u0[:C12e] = 0.0
    prob = ODEProblem(SIMPLIFIED_REACTION_NETWORK, u0, (0.0, tmax), p; sparse=true)
    sol = solve(prob, Rodas5P(); reltol=1e-12, abstol=1e-12, save_everystep=false)
    u = sol[end]
    return u[10] - u[9], u
end

# find sign change on a grid, then refine with bisection
function switch_c6e_for_dm(dm; c6min=0.0, c6max=1.0, ngrid=100, tol=1e-8)
    
    first_grid = range(1e-6, 1e-5; length=ngrid)
    second_grid = range(1e-5, 1e-4; length=ngrid)
    third_grid = range(1e-4, 1e-3; length=1000)
    fourth_grid = range(1e-3, 1e-2; length=1000)
    fifth_grid = range(1e-2, 1e-1; length=1000)
    sixth_grid = range(1e-1, 1.0; length=ngrid)
    # Join the grids
    c6_grid = vcat(collect(first_grid), collect(second_grid), collect(third_grid), collect(fourth_grid), collect(fifth_grid), collect(sixth_grid))
    # Remove possible duplicates
    c6_grid = unique(c6_grid)

    vals = [(c, delta_at_c6e(c; dm=dm)[1]) for c in c6_grid]
    idx = findfirst(i -> vals[i][2]*vals[i+1][2] < 0, 1:length(vals)-1)
    isnothing(idx) && return (dm=dm, c6e_star=missing, u_star=missing, note="no sign change in range")
    a, b = vals[idx][1], vals[idx+1][1]
    fa = vals[idx][2]
    fb = vals[idx+1][2]
    for _ in 1:60
        m = (a+b)/2
        fm, _ = delta_at_c6e(m; dm=dm)
        if abs(fm) < tol || (b-a) < tol
            c6e_star = m
            _, u_star = delta_at_c6e(c6e_star; dm=dm)
            return (dm=dm, c6e_star=c6e_star, u_star=u_star, note="ok")
        end
        if fa*fm <= 0
            b = m; fb = fm
        else
            a = m; fa = fm
        end
    end
    c6e_star = (a+b)/2
    _, u_star = delta_at_c6e(c6e_star; dm=dm)
    return (dm=dm, c6e_star=c6e_star, u_star=u_star, note="max iter hit")
end

# one-shot wrapper over the 10 Dm values
function find_switches(dm_list)
    [switch_c6e_for_dm(dm) for dm in dm_list]
end

# run it
Dm_grid = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0, 3.0, 5.0, 10.0]
switch_results = find_switches(Dm_grid)

# print summary
for r in switch_results
    println("\\nDm = ", r.dm, "   status: ", r.note)
    if r.c6e_star === missing
        continue
    end
    u = r.u_star
    println("  C6e* ≈ ", r.c6e_star)
    println("  GFP = ", u[10], "   RFP = ", u[9], "   Δ = ", u[10]-u[9])
end


\nDm = 0.01   status: ok
  C6e* ≈ 0.011479302483635982
  GFP = 0.050063285857200115   RFP = 19.741848498559825   Δ = -19.691785212702626
\nDm = 0.05   status: ok
  C6e* ≈ 0.002377898894988739
  GFP = 0.05006227906159225   RFP = 19.741741194077782   Δ = -19.69167891501619
\nDm = 0.1   status: ok
  C6e* ≈ 0.0012412329321509008
  GFP = 0.05006209739163216   RFP = 19.741726941962835   Δ = -19.691664844571203
\nDm = 0.25   status: ok
  C6e* ≈ 0.0005615533502252251
  GFP = 0.05006200050845013   RFP = 19.741717867175492   Δ = -19.69165586666704
\nDm = 0.5   status: ok
  C6e* ≈ 0.00033788358671171174
  GFP = 20.049891573600448   RFP = 0.01001255774780623   Δ = 20.03987901585264
\nDm = 0.75   status: ok
  C6e* ≈ 0.0002648754222972973
  GFP = 0.050061948286098626   RFP = 19.74171403025188   Δ = -19.69165208196578
\nDm = 1.0   status: ok
  C6e* ≈ 0.00022903645833333334
  GFP = 20.049891572902865   RFP = 0.010012523798415607   Δ = 20.03987904910445
\nDm = 3.0   status: ok
  C6e* ≈ 0.00015970228040